In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.dq_checks import dq_validate_checks

In [2]:
from pyspark.sql.functions import current_timestamp, current_date, lit, expr, to_date, when, lower, upper, trim, concat_ws, xxhash64, cast, col, lead, date_sub, coalesce, date_format
from pyspark.sql.window import Window

In [3]:
spark = get_sparkSession("Load gold.fact_customer_events")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp
from delta import DeltaTable

In [6]:
_schema_name = "gold"
_table_name = "fact_customer_events"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "gold_fact_customer_events.ipynb"
_silver_table = "silver.fact_customer_events"
_bad_table = f"bad_{_table_name}"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for gold.fact_customer_events


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Reading from silver.customer and generating surrogate key and business key

df = spark.read.table(_silver_table)

print(f"SPARK-APP: Silver Table Data count : {df.count()}")
df.show(truncate = False)

SPARK-APP: Silver Table Data count : 7
+--------------------------+---------------------------------+--------------------------+---------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+
|created_on                |created_by                       |modified_on               |modified_by                      |source_file             |product_id|product_category|event_id|customer_id|channel|store |event_type      |event_timestamp    |city     |state|country|device_os|device_type|session_id|payment_method|
+--------------------------+---------------------------------+--------------------------+---------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+-----------

In [9]:
# DQ validations

_row = df \
        .select("source_file") \
        .distinct() \
        .limit(1) \
        .first()

_source_file = "UNKNOWN" if _row is None else _row[0]

df = dq_validate_checks(df, spark, _schema_name, _table_name, _source_file)

print("SPARK-APP: DQ validations completed")
print(f"SPARK-APP: Table Data count after DQ validations : {df.count()}")

SPARK-APP: DQ validations completed
SPARK-APP: Table Data count after DQ validations : 7


In [10]:
#Adding audit columns

df = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name))

print(f"SPARK-APP: Silver Table Data count : {df.count()}")

# Generating surrogate key and business key

df_sk = df.withColumn("event_sk", xxhash64(concat_ws("||", "event_id", coalesce("product_id", lit("NA")))).cast("bigint")) \
          .withColumn("customer_bk", xxhash64(concat_ws("||", "customer_id", "store", "channel")).cast("bigint"))

df_sk.show(truncate = False)

SPARK-APP: Silver Table Data count : 7
+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+--------------------+--------------------+
|created_on                |created_by                     |modified_on               |modified_by                    |source_file             |product_id|product_category|event_id|customer_id|channel|store |event_type      |event_timestamp    |city     |state|country|device_os|device_type|session_id|payment_method|event_sk            |customer_bk         |
+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+----------------

In [11]:
# Join with other dimension tables to get their surrogate keys

df_gold_store = spark.read.table("gold.dim_store").select("store_sk", "store_code")
df_gold_channel = spark.read.table("gold.dim_channel").select("channel_sk", "channel_code")
df_gold_customer = spark.read.table("gold.dim_customer").where("is_current = true").select("customer_sk", "customer_bk")

df_invalid = df_sk.join(df_gold_customer, how = 'left_anti', on=df_sk.customer_bk==df_gold_customer.customer_bk)

print(f"SPARK-APP: Bad Record Count : {df_invalid.count()}")  

df_invalid.show(truncate = False)

if df_invalid.count() > 0:
    df_invalid.withColumn('error_description', lit("Customer not found in gold.dim_customer")) \
                       .select("event_id", "customer_id", "store", "channel", "event_type", "event_timestamp", 
                               "city", "state", "country", "device_type", "device_os", "session_id", "product_id", 
                               "product_category", "payment_method", "source_file", "created_on", "created_by", "error_description") \
                       .write \
                       .format("delta") \
                       .mode("append") \
                       .saveAsTable(f"bad.{_bad_table}")

df_silver = df_sk.join(df_gold_store, how="inner", on=df_sk.store==df_gold_store.store_code) \
                 .join(df_gold_channel, how="inner", on=df_sk.channel==df_gold_channel.channel_code) \
                 .join(df_gold_customer, how = "inner", on=df_sk.customer_bk==df_gold_customer.customer_bk) \
                 .select("event_sk", "event_id", "customer_sk", "product_id", "product_category", "event_type", "event_timestamp", 
                         "store_sk", "channel_sk", "device_type", "device_os", "city", "state", "country", "payment_method", 
                         "session_id", "source_file", "created_on", "created_by", "modified_on", "modified_by")

print(f"SPARK-APP: Total records to be written to {_schema_name}.{_table_name} : {df_silver.count()}")

df_silver.show(truncate = False)

SPARK-APP: Bad Record Count : 2
+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+-------------------+--------------------+
|created_on                |created_by                     |modified_on               |modified_by                    |source_file             |product_id|product_category|event_id|customer_id|channel|store |event_type      |event_timestamp    |city     |state|country|device_os|device_type|session_id|payment_method|event_sk           |customer_bk         |
+--------------------------+-------------------------------+--------------------------+-------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+-----

In [12]:
spark.read.table(f"bad.{_bad_table}").show(truncate = False)

+--------+-----------+------+-------+----------------+-------------------+---------+-----+-------+-----------+---------+----------+----------+----------------+--------------+--------------------------+-------------------------------+------------------------+---------------------------------------+
|event_id|customer_id|store |channel|event_type      |event_timestamp    |city     |state|country|device_type|device_os|session_id|product_id|product_category|payment_method|created_on                |created_by                     |source_file             |error_description                      |
+--------+-----------+------+-------+----------------+-------------------+---------+-----+-------+-----------+---------+----------+----------+----------------+--------------+--------------------------+-------------------------------+------------------------+---------------------------------------+
|EVT0002 |CUST002    |AMZ_US|MKT    |add_to_cart     |2025-01-10 10:18:10|Seattle  |WA   |US     |MOBIL

In [13]:
# Truncate table for full load
dt_fact_cust_events = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")

if get_max_loadTimestamp(spark, _schema_name, _table_name) == '1900-01-01 00:00:00.000000':
    
    #Full-load so truncate dim table
    spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled=false")
    
    dt_fact_cust_events.delete()
    dt_fact_cust_events.vacuum(0)


In [14]:
# Merge
dt_fact_cust_events.alias("t").merge(
    df_silver.alias("s"),
    "t.event_sk = s.event_sk"
).whenNotMatchedInsertAll().execute()
                     
print("SPARK-APP: Merge completed") 

SPARK-APP: Merge completed


In [15]:
hist = dt_fact_cust_events.history().limit(1).select("version", "operationMetrics.executionTimeMs",
                                          "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated",
                                          "operationMetrics.numOutputRows")

hist.show()
row = hist.first()

loaded_count = int("0" if row is None else row["numTargetRowsInserted"]) + int("0" if row is None else row["numTargetRowsUpdated"])


+-------+---------------+---------------------+--------------------+-------------+
|version|executionTimeMs|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+---------------+---------------------+--------------------+-------------+
|      1|           5952|                    5|                   0|            5|
+-------+---------------+---------------------+--------------------+-------------+



In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+--------+--------------------+----------+----------------+------------+-------------------+--------+----------+-----------+---------+-------+-----+-------+--------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
|            event_sk|event_id|         customer_sk|product_id|product_category|  event_type|    event_timestamp|store_sk|channel_sk|device_type|device_os|   city|state|country|payment_method|session_id|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+--------------------+--------+--------------------+----------+----------------+------------+-------------------+--------+----------+-----------+---------+-------+-----+-------+--------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 7626088190278455038| EVT0001| -798410774282725028|     P1001|     Elect

In [17]:
#Writing log data to load_controller

_data = []

_data.append([_schema_name, _schema_name, _table_name, "delta-load", "merge", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to load_controller")

SPARK-APP: Data written to load_controller


In [18]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|layer|schema_name|          table_name| load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| gold|       gold|fact_customer_events|delta-load|     merge|2026-01-29 03:52:...|    success|           5|2026-01-29 03:52:...|gold_fact_custome...|2026-01-29 03:52:...|gold_fact_custome...|
+-----+-----------+--------------------+----------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [19]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, f"{_schema_name}.{_table_name}")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

SPARK-APP: symlink manifest file generated


In [20]:
spark.stop()